# Ordered Logistic Regression Results for Adoption Predictors – Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json), which covers ordered logistic regression results for predictors of indigenous and modern knowledge adoption in rangeland management, using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
Defined by a Croissant schema and accessible at the following URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Install mlcroissant
!pip install -U mlcroissant

## 1. Data Loading

Load dataset metadata and inspect the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Name: ', metadata.name)
print('\nDescription:')
print(metadata.description)


## 2. Data Overview

List all record sets and fields available, along with their `@id`s. This helps identify what tables and columns are defined for data access and processing.

In [ ]:
# List all available record sets and their IDs
record_sets = dataset.record_sets
if not record_sets:
    print('No record sets are defined in the top-level metadata.')
else:
    for rs in record_sets:
        print(f"Record Set: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id})")
        print()
# If record_sets is empty, we search for any records via preview
if not record_sets:
    print("No explicit record sets found. Trying to discover record sets from available resources...")
    # Show distributions
    if hasattr(metadata, 'distribution'):
        distributions = metadata.distribution
        print(f"Distributions: {[d['@id'] if isinstance(d, dict) and '@id' in d else d for d in distributions]}")


## 3. Data Extraction

Load tabular record set(s) into pandas DataFrame(s) for analysis. Use the record set and field `@id`s for access. Since this dataset is a Croissant-packaged research output, there may be one or more record sets referencing data tables (e.g., regression outputs, survey tables, etc.).

Below, we enumerate available record sets, extract their data, and display the first rows and column IDs.

In [ ]:
# Attempt to list all record sets (using their IDs) and extract them
from collections.abc import Iterable

# We may need to infer available ids, since record_sets might be empty.
rs_ids = [rs.id for rs in dataset.record_sets] if dataset.record_sets else []

# If no record sets found, try to infer from available methods
dataframes = {}

if not rs_ids:
    # Try to list available record set IDs by inspecting dataset.records()
    preview_rs_ids = list(dataset._preview['record_sets'].keys()) if hasattr(dataset, '_preview') and 'record_sets' in dataset._preview else []
    if preview_rs_ids:
        rs_ids = preview_rs_ids

if rs_ids:
    print("Found record set IDs:", rs_ids)
    for record_set_id in rs_ids:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nDataFrame for record set '@id': {record_set_id}")
            print("Columns (fields by @id):", df.columns.tolist())
            display(df.head())
        else:
            print(f"No records for record set @id: {record_set_id}")
else:
    print("No record sets found in the dataset definition.")
    print("If the dataset defines only files or distributions, please refer to those for further processing.")

## 4. Exploratory Data Analysis (EDA)

We will demonstrate typical preprocessing tasks: filtering, normalization, grouping, and previewing descriptive statistics. Replace `<record_set_id>` and `<numeric_field_id>` with concrete values as discovered above.

*If no record sets are provided, this section may not run and should be adapted to the dataset structure.*

In [ ]:
# Example: Use the first available record set for EDA (if any)
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using record set '@id': {record_set_id}")
    
    # Show numeric columns by simple dtype inspection
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_columns:
        # Try to infer numeric columns by attempting to convert
        for col in df.columns:
            try:
                pd.to_numeric(df[col])
                numeric_columns.append(col)
            except Exception:
                continue
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        print(f"Numeric field (@id): {numeric_field_id}")

        # Attempt filtering for values above mean (or 0)
        try:
            vals = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = vals.mean() if vals.count() > 0 else 0
            filtered_df = df[vals > threshold].copy()
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
            display(filtered_df.head())

            filtered_df[f"{numeric_field_id}_normalized"] = (vals - vals.mean()) / vals.std()
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Find a non-numeric column to group by
            non_numeric = [c for c in df.columns if c != numeric_field_id]
            group_field = None
            for c in non_numeric:
                if df[c].nunique() < len(df) // 2:
                    group_field = c
                    break
            if group_field:
                # Group and show 
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
                print(f"\nGrouped means of {numeric_field_id} by {group_field}:")
                display(grouped_df.head())
        except Exception as e:
            print("Error in EDA:", e)
    else:
        print("No numeric fields found for analysis.")
else:
    print("No DataFrame extracted for EDA.")

## 5. Visualization

Plot basic visualizations (histograms, boxplots, etc.) for selected fields. You may need to adjust field names/IDs based on earlier output. If there is no structured data, skip this section.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        # Histogram
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()
        # Boxplot
        plt.figure(figsize=(4,6))
        sns.boxplot(y=df[numeric_field_id].dropna())
        plt.title(f"Boxplot of {numeric_field_id}")
        plt.ylabel(numeric_field_id)
        plt.show()
    else:
        print("No numeric columns found for plotting.")
else:
    print("No data extracted for visualization.")

## 6. Conclusion

Using `mlcroissant`, we explored the FAIR^2 dataset for rangeland management interventions, covering ordered logistic regression outputs in Northern Kenya. We demonstrated how to load, inspect, filter, and visualize the dataset. For domain-specific analyses, tailor the EDA and visualizations to variables of interest as identified in the field/extraction overview.

_You can further extend this notebook to apply more advanced analysis, integrate with ML workflows, and automate dataset exploration for similarly structured Croissant datasets._